# Biblioteka Bokeh

Bokeh to biblioteka wyróżniająca się wysokim poziomem interaktywności. Umożliwia dostosowywanie wizualizacji w czasie rzeczywistym  dla użytkowników, którzy nie mają styczności z kodem.

In [1]:
from bokeh.io import output_notebook, reset_output, show
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource,
    CrosshairTool,
    LassoSelectTool,
    PolySelectTool,
    WheelZoomTool,
    PanTool,
    HoverTool,
    UndoTool,
    RedoTool,
    ResetTool,
    SaveTool,
)
from bokeh.layouts import gridplot
from bokeh.transform import dodge

reset_output()
output_notebook() # ouput zapisany w jupyter

Loading BokehJS ...

In [2]:
import numpy as np
import pandas as pd
df = pd.read_csv("athlete_events_pd.csv")

--------------------

##### ⭐ Zadanie 1: 

Przygotuj wykres punktowy (`circle`) opierając się na danych, które sam wybierzesz w sensowny sposób. W rozwiązaniu zdefiniuj własny pasek narzędzi składający się z narzędzi:

- `LassoSelectTool` i `xPanTool` z kategorii narzędzi *Gestures* (*Pan/Drag tools*),
- `PolySelectTool` z kategorii narzędzi *Gestures* (*Click/Tap tools*),
- `WheelZoomTool` z kategorii narzędzi *Gestures* (*Scroll/Pinch tools*),
- `UndoTool`, `RedoTool`, `ResetTool` i `SaveTool` z kategorii narzędzi *Actions*,
- `HoverTool` z etykietami danych z kategorii narzędzi *Inspectors*.

Zapoznaj się z parametrem `toolbar_location` i dobierz dla niego nową wartość. Zapoznaj się z metodą `autohide` obiektu `toolbar` i dobierz dla niego taką wartość, żeby pasek narzędzi chował się po zjechaniu kursorem myszy z wizualizacji. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [3]:
df_clean = df[["Height", "Weight", "Name", "Sport", "Team", "Age"]].dropna()
df_sample = df_clean.sample(n=10_000)
source = ColumnDataSource(df_sample)
lasso = LassoSelectTool()
xpan = PanTool(dimensions='width')
ypan = PanTool(dimensions='height')
poly = PolySelectTool()
wheel_zoom = WheelZoomTool()
hover = HoverTool(tooltips=[
    ('Nazwisko', '@Name'),
    ('Wiek (lata)', '@Age'),
    ('Waga (kg)', '@Weight'),
    ('Sport', '@Sport'),
    ('Kraj/Zespół', '@Team')
])
tools = [lasso, xpan, ypan, poly, wheel_zoom, UndoTool(), RedoTool(), ResetTool(), SaveTool(), hover]
p = figure(title='Zależność wzrostu od wagi sportowców olimpijskich', width=900, height=600, x_axis_label='Waga (kg)', y_axis_label='Wzrost (cm)', tools='', toolbar_location='below')
p.add_tools(*tools)
p.scatter(
    x="Weight",
    y="Height",
    source=source,
    size=6,
    marker="circle",
    alpha=0.6,
)
p.toolbar.autohide=True
show(p)

##### ⭐ Zadanie 2:

Przygotuj wykres kolumnowy (`vbar`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Każda seria danych musi być przedstawiona na osobnym podwykresie (`subplot`) jednego obrazu. Zapoznaj się z metodami `column`, `row` i `gridplot` oraz dobierz dla nich właściwą wartość mając na uwadze wizualizację określonej liczby serii danych na osobnych podwykresach. Dodaj powiązane zachowania pomiędzy podwykresami:

- współdzielony ruchomy zakres dla wartości na osi X, także przy poziomym przesuwaniu wykresu,
- współdzielony wskaźnik będący narzędziem `CrosshairTool` widoczny jednocześnie na wszystkich podwykresach.

Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i ewentualnie legenda). 

In [5]:
medals = df[df["Medal"].notna()]
countries_eng = ['Denmark', 'Norway', 'Sweden', 'Finland', 'Iceland']
countries_pl  = ['Dania', 'Norwegia', 'Szwecja', 'Finlandia', 'Islandia']

medal_types = ['Gold', 'Silver', 'Bronze']
medal_labels = {'Gold': 'Złote medale', 'Silver': 'Srebrne medale', 'Bronze': 'Brązowe medale'}
colors = {'Gold': 'gold', 'Silver': 'silver', 'Bronze': '#cd7f32'}

medal_counts = {
    medal: (medals.loc[medals["Medal"] == medal, "Team"]
                  .value_counts()
                  .reindex(countries_eng, fill_value=0)
                  .tolist())
    for medal in medal_types
}

x_range = countries_pl 
crosshair = CrosshairTool(dimensions="both")
plots = []

for medal in medal_types:
    source = ColumnDataSource(data=dict(
        country=countries_pl,
        count=medal_counts[medal],
    ))

    p = figure(
        x_range=x_range,
        width=900, height=250,
        tools="xpan,xwheel_zoom,reset,save",
        toolbar_location=None,
        title=medal_labels[medal] + " – kraje nordyckie"
    )

    p.vbar(x="country", top="count", width=0.6, source=source, fill_color=colors[medal])
    p.xaxis.axis_label = "Kraj"
    p.yaxis.axis_label = "Liczba medali"
    p.xgrid.grid_line_color = None
    p.y_range.start = 0
    p.title.text_font_size = "14pt"
    p.add_tools(crosshair)
    plots.append(p)

for plt in plots[1:]:
    plt.x_range = plots[0].x_range

layout = gridplot(plots, ncols=1, merge_tools=True, toolbar_location="above")
show(layout)

##### ⭐ Zadanie 3:

Przygotuj wykres liniowy (`line`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Zapoznaj się z parametrem `location` obiektu `legend` i dobierz dla niego nową wartość. Zapoznaj się z parametrem `click_policy` obiektu `legend` i dobierz dla niego taką wartość, żeby po kliknięciu w tytuł danej serii danych w legendzie, jej wizualizacji znikała z wykresu. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i legenda). 

In [6]:
medals = df[df["Medal"].notna()] 
countries_eng = ['Denmark', 'Norway', 'Sweden', 'Finland', 'Iceland']
countries_pl  = ['Dania', 'Norwegia', 'Szwecja', 'Finlandia', 'Islandia']
country_translate = dict(zip(countries_eng, countries_pl))
grouped = medals.groupby(["Year", "Team"]).size().unstack(fill_value=0)
grouped = grouped.reindex(columns=countries_eng, fill_value=0)
grouped = grouped.reset_index()

source = ColumnDataSource(grouped)

p = figure(
    x_axis_type="linear",
    width=1000,
    height=800,
    title="Suma wszystkich medali w czasie dla krajów nordyckich",
    toolbar_location="above",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for idx, country in enumerate(countries_eng):
    p.line(x='Year', y=country, source=source, line_width=2, color=colors[idx], legend_label=country_translate[country])

p.xaxis.axis_label = "Rok"
p.yaxis.axis_label = "Liczba medali"

p.title.text_font_size = "16pt"
p.xaxis.axis_label_text_font_size = "12pt"
p.yaxis.axis_label_text_font_size = "12pt"
p.xaxis.major_label_text_font_size = "11pt"

p.legend.location = "top_right"            
p.legend.title = "Kraj"
p.legend.label_text_font_size = "11pt"
p.legend.title_text_font_size = "12pt"
p.legend.click_policy = "hide"     

show(p)

##### ⭐ Zadanie 4:

Przedstaw na dowolnym wykresie dowolne zestawienie danych przygotowane na podstawie pliku `athlete_events.csv` z pierwszego tygodnia. Zadbaj o czytelność wykresu. Do swojej wizualizacji dodaj co najmniej 3 różne widgety, które będą miały na nią wpływ:

- `AutocompleteInput`,
- `Button`,
- `CheckboxButtonGroup`,
- `CheckboxGroup`,
- `ColorPicker`,
- `DataCube`,
- `DataTable`,
- `DatePicker`,
- `DateRangePicker`,
- `MultipleDatePicker`,
- `DatetimePicker`,
- `DatetimeRangePicker`,
- `MultipleDatetimePicker`,
- `TimePicker`,
- `DateRangeSlider`,
- `DateSlider`,
- `DatetimeRangeSlider`,
- `Div`,
- `Dropdown`,
- `FileInput`,
- `MultiChoice`,
- `MultiSelect`,
- `NumericInput`,
- `Paragraph`,
- `PasswordInput`,
- `PreText`,
- `RadioButtonGroup`,
- `RadioGroup`,
- `RangeSlider`,
- `Select`,
- `Slider`,
- `Spinner`,
- `Switch`,
- `Tabs`,
- `TextAreaInput`,
- `TextInput`,
- `Toggle`.

Dodatkowo, zastosuj `HelpButton` z `Tooltip` lub dodaj `Tooltip` do przynajmniej 1 widgetu.